# コエコミ — Colab バックエンド起動

このノートブックを **1台のColab = 1サーバー** として実行します。
本番は **3台**（Colab Pro+）。`SERVER_ID` と `SERVER_COLOR` を台ごとに変えてください。

> **3台あるのは速さのためではなく、1台落ちても誰も気づかないためです。**
> 10人・3台なら GPU は余ります。負荷分散はしていません（iPad 側が
> 端末IDのハッシュで散らし、死んでいたら次の台に移ります）。

**⚠️ ランタイムは GPU にしてください**（ランタイム > ランタイムのタイプを変更 > GPU）。
GPU 無しで動作確認だけしたい場合は、セル2で `TTS_BACKEND='dummy'` にします。

**⚠️ イベントが終わったら必ずランタイムを停止してください。**
Pro+ のバックグラウンド実行はタブを閉じても動き続け、コンピューティングユニットを消費します。

| ID | color | label |
|----|-------|-------|
| colab-1 | red | 赤サーバー |
| colab-2 | blue | 青サーバー |
| colab-3 | green | 緑サーバー |

In [ ]:
# 1) リポジトリを取得（2回目以降は最新に更新）
import os

if os.path.isdir("/content/koekomi/.git"):
    !cd /content/koekomi && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/TaiyoYamada/koekomi.git /content/koekomi
%cd /content/koekomi
!git log --oneline -1

In [ ]:
# 2) 設定（秘密情報は Colab の「シークレット」から読み込む。直書きしない）
import os
from google.colab import userdata  # 左の鍵アイコンから登録

# 名簿（今日のURLを配る先）
os.environ["GAS_URL"] = userdata.get("GAS_URL")

# イベントの合言葉。フロントの VITE_EVENT_TOKEN と同じ文字列にすること。
# 未設定だと誰でもこのAPIを叩けます（子どもの声を扱うので必ず設定）。
os.environ["EVENT_TOKEN"] = userdata.get("EVENT_TOKEN")

# フロントの公開オリジン。写真の取得元であり、CORS の許可先でもある。
# 未設定だと動画をサーバーで作れず、iPad 側の書き出し（実時間）になります。
os.environ["FRONTEND_ORIGIN"] = "https://koekomi.vercel.app"  # ← 自分のURLに変える

# AIバックエンド: 既定で Qwen3-TTS（声クローン。GPU必須）。
# GPUが無い／動作確認だけなら次の行を有効にする:
# os.environ['TTS_BACKEND'] = 'dummy'

# この台の識別情報（台ごとに変える）
os.environ["SERVER_ID"] = "colab-1"
os.environ["SERVER_COLOR"] = "red"
os.environ["SERVER_LABEL"] = "赤サーバー"

# GPU 1枚なら 1 のまま。増やしても TTS_SERIALIZE=1 の間は GPU 並列度は 1。
os.environ["WORKERS"] = "1"

In [ ]:
# 3) 起動（依存インストール → FastAPI起動 → トンネル公開 → 名簿登録 → 自己チェック → heartbeat）
#    このセルは実行したままにしておく（Pro+ のバックグラウンド実行ならタブを閉じてもOK）。
%run colab/colab_runner.py

## 当日の確認

起動ログの最後に出る URL を使って、手元のPCから通しで確認します。

```bash
bash scripts/smoke-test.sh https://xxxx.trycloudflare.com <EVENT_TOKEN>
```

`/health` → `/voices` → `/jobs` → `/artifacts` → `/render` まで、
子どもがやるのと同じ順序で通します。**全部 PASS してから会場を開けてください。**
1行あたりの生成時間も表示されるので、待ち時間の見積もりにも使えます。